# Maize Tassel Detection - YOLO26s Training
**FYP-26-S2-7 | Adapted Colab Notebook**

## Before You Start
1. Upload  and  to your Google Drive root folder
2. Run each cell in order - do NOT skip cells
3. Training takes 4-6 hours on Tesla T4 GPU (free Colab)
4. Final model saved to  on your Drive


Maize_YOLO26_Final.ipynb

# 🌽 Maize Tassel Counter — YOLO26s Final
### MCD (Ground, real .mat bboxes) + MTDC-UAV (Drone, .xml bboxes)
### 🛡️ Crash-proof edition — auto Drive backup + full resume support

---
### ⚠️  BEFORE ANYTHING: Enable GPU
> **Runtime → Change runtime type → T4 GPU → Save**

---
### 🗺️ How to use this notebook
| Situation | What to run |
|---|---|
| **Fresh start** | Cell 1 → Cells 2–9 in order |
| **Runtime crashed mid-training** | Cell 1 → Cell 2 → **Cell 2b** → Cell R |
| **Free-tier limit hit (12hr)** | Cell 1 → Cell 2 → **Cell 2b** → Cell R |
| **Want to eval / infer after training** | Cell 1 → Cell 2 → Cell 2b → Cells 10-12 |
| **Save everything to Drive** | Cell 13 |

---
### 📋 Pipeline
1. ✅ GPU check & install (single cell)
2. 📁 Mount Drive, extract archives, configure paths
3. 🛡️ **Cell 2b** — crash-proof backup system (REQUIRED before training)
4. 🔍 Validate dataset paths
5. 🔄 Parse annotations (MCD .mat real bboxes, UAV .xml → YOLO)
6. ✂️ SAHI tile UAV images
7. 📦 Build final dataset
8. 🚀 Stage 1 — backbone frozen, ground warmup (50 epochs)
9. 🚀 Stage 2 — all layers, combined data (80 epochs)
10. 🚀 Stage 3 — fine-tune at 1280px (80 epochs)
11. 📈 Training curves
12. 📊 Evaluation
13. 🔍 Inference
14. 💾 Save to Drive
15. 🔁 **Cell R** — Resume after crash

---
## ✅ Cell 1 — Install All Packages
*Run this every time you (re)connect.*

> **Cell 1a has been removed.** The forced NumPy downgrade + kernel restart
> was a legacy workaround. `numpy<2` is now pinned directly in the install
> command below, which is sufficient. No restart is needed.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 1 — Install All Packages  (run every time you connect)    ║
# ╚══════════════════════════════════════════════════════════════════╝
import subprocess, sys

r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print('✅ GPU detected!')
else:
    print('❌ NO GPU — Runtime → Change runtime type → T4 GPU → Save')
    raise SystemExit('No GPU attached')

subprocess.run(['apt-get', 'install', 'unrar', '-y', '-q'], capture_output=True)
if subprocess.run(['which', 'unrar'], capture_output=True).returncode != 0:
    subprocess.run(['apt-get', 'install', 'unrar-free', '-y', '-q'])

print('Installing packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'ultralytics>=8.3.0,<9.0',
    'scipy', 'sahi', 'tqdm', 'matplotlib',
    'pillow', 'pyyaml', 'opencv-python-headless',
    '-q'], check=True)

import torch, ultralytics
import numpy as np

print(f'\n✅ PyTorch       : {torch.__version__}')
print(f'✅ CUDA          : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'✅ GPU           : {g.name}  ({g.total_memory/1e9:.1f} GB VRAM)')
print(f'✅ ultralytics   : {ultralytics.__version__}')
print(f'✅ numpy         : {np.__version__}')
print(f'✅ packages installed — ready to proceed')


✅ GPU detected!
Installing packages...
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

✅ PyTorch       : 2.11.0+cu128
✅ CUDA          : True
✅ GPU           : Tesla T4  (15.6 GB VRAM)
✅ ultralytics   : 8.4.60
✅ numpy         : 2.0.2
✅ packages installed — ready to proceed


---
## 📁 Cell 2 — Mount Drive, Extract & Configure
*Run this every time you (re)connect.*

**Your Google Drive must have:**
```
MyDrive/
├── Copy of MTDC-UAV.rar
└── Copy of maize_counting_dataset.zip
```


In [ ]:
# drive.flush_and_unmount()

In [2]:
from google.colab import drive
import os, zipfile, subprocess
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:


# ── ✏️  EDIT THESE IF YOUR FILENAMES DIFFER ──────────────────────────────────
UAV_RAR_PATH  = '/content/drive/MyDrive/MTDC-UAV.zip'
MCD_ZIP_PATH  = '/content/drive/MyDrive/maize_counting_dataset.zip'
DRIVE_RESULTS = '/content/drive/MyDrive/maize_yolo26_results'

# ── TRAINING CONFIG ───────────────────────────────────────────────────────────
# NOTE: POINT_BOX_SIZE removed — MCD dataset has REAL bounding boxes in
#       annotation['bndbox'] stored as [x, y, width, height]. These are now
#       used directly; no synthetic pseudo-boxes are created.
MODEL_SIZE     = 'yolo26s'  # yolo26n | yolo26s | yolo26m | yolo26l | yolo26x
IMG_SIZE       = 640
BATCH          = 16         # reduce to 8 if you get CUDA OOM errors
# YOLO26 eliminates IoU-based NMS (the one-to-one head resolves duplicates during training).
# Confidence filtering still occurs internally, and our cross-tile merge (Cell 12) still
# applies a manual IoU NMS pass in full-image space — NMS_IOU is used there.
NMS_IOU        = 0.4
TRAIN_IOU      = 0.7        # IoU threshold for STAL label assignment during training (YOLO26 default)
                             # ⚠️  Do NOT lower — 0.7 matches YOLO26's one-to-one head training regime
EVAL_CONF_THR  = 0.30       # confidence threshold for eval & counting
RUN_NAME       = 'maize_yolo26'
# ─────────────────────────────────────────────────────────────────────────────

EXTRACT_PATH = '/content/dataset'
WORK_DIR     = '/content/maize_work'
DATASET_DIR  = os.path.join(WORK_DIR, 'dataset')
TILED_DIR    = os.path.join(WORK_DIR, 'uav_tiled')
GROUND_DIR   = os.path.join(WORK_DIR, 'dataset_ground_only')

# Stage weight paths
S1_NAME      = f'{RUN_NAME}_s1'
S2_NAME      = f'{RUN_NAME}_s2'
S3_NAME      = f'{RUN_NAME}_s3_final'
S1_BEST      = f'{WORK_DIR}/runs/detect/{S1_NAME}/weights/best.pt'
S2_BEST      = f'{WORK_DIR}/runs/detect/{S2_NAME}/weights/best.pt'
BEST_WEIGHTS = f'{WORK_DIR}/runs/detect/{S3_NAME}/weights/best.pt'
YAML_PATH    = os.path.join(DATASET_DIR, 'dataset.yaml')
GROUND_YAML  = os.path.join(GROUND_DIR,  'dataset.yaml')

for d in [EXTRACT_PATH, WORK_DIR, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)
os.chdir(WORK_DIR)

# ── Extract UAV RAR ────────────────────────────────────────────────────────
UAV_EXTRACT = os.path.join(EXTRACT_PATH, 'uav')
MCD_EXTRACT = os.path.join(EXTRACT_PATH, 'mcd')
os.makedirs(UAV_EXTRACT, exist_ok=True)
os.makedirs(MCD_EXTRACT, exist_ok=True)

if os.path.exists(UAV_RAR_PATH):
    print('Extracting UAV .rar…')
    r = subprocess.run(['unrar','x','-o+', UAV_RAR_PATH, UAV_EXTRACT+'/'],
                       capture_output=True, text=True)
    print('✅ UAV extracted' if r.returncode==0 else f'❌ unrar failed:\n{r.stderr[:400]}')
else:
    print(f'❌ Not found: {UAV_RAR_PATH}')

if os.path.exists(MCD_ZIP_PATH):
    print('Extracting MCD .zip…')
    try:
        with zipfile.ZipFile(MCD_ZIP_PATH, 'r') as z:
            z.extractall(MCD_EXTRACT)
        print('✅ MCD extracted')
    except zipfile.BadZipFile as e:
        print(f'❌ ZIP corrupt: {e}')
else:
    print(f'❌ Not found: {MCD_ZIP_PATH}')

print('\n✅ Cell 2 done — run Cell 2b next (REQUIRED before training)')


Extracting UAV .rar…
✅ UAV extracted
Extracting MCD .zip…
✅ MCD extracted

✅ Cell 2 done — run Cell 2b next (REQUIRED before training)


---
## 🛡️ Cell 2b — Crash-Proof Backup System
**⚠️ ALWAYS run this before starting or resuming training.**


In [4]:
import shutil, os, threading, time, json
from ultralytics import YOLO

BACKUP_EVERY_N_EPOCHS = 5

STATUS_FILE = os.path.join(DRIVE_RESULTS, 'CHECKPOINT_STATUS.txt')

def write_status(stage, epoch, note=''):
    try:
        with open(STATUS_FILE, 'w') as f:
            f.write(f'stage={stage}\n')
            f.write(f'epoch={epoch}\n')
            f.write(f'note={note}\n')
            f.write(f'time={time.strftime("%Y-%m-%d %H:%M:%S")}\n')
    except Exception:
        pass

def on_train_epoch_end(trainer):
    epoch      = trainer.epoch + 1
    run_dir    = trainer.save_dir
    stage_label = os.path.basename(str(run_dir))

    if epoch % BACKUP_EVERY_N_EPOCHS == 0:
        for fname in ['last.pt', 'best.pt']:
            src = os.path.join(run_dir, 'weights', fname)
            if os.path.exists(src):
                dst_specific = os.path.join(DRIVE_RESULTS, f'{stage_label}_{fname}')
                shutil.copy2(src, dst_specific)
                if fname == 'last.pt':
                    shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_last.pt'))
                if fname == 'best.pt':
                    shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_best.pt'))
        write_status(stage_label, epoch, 'epoch-end backup OK')
        print(f'  💾 [Epoch {epoch}] Backed up → {DRIVE_RESULTS}')

_BACKUP_CALLBACKS = {'on_train_epoch_end': on_train_epoch_end}
print(f'✅ Epoch callback ready (every {BACKUP_EVERY_N_EPOCHS} epochs)')

_stop_bg = threading.Event()

def _bg_backup_loop():
    while not _stop_bg.is_set():
        time.sleep(600)
        try:
            runs_root = os.path.join(WORK_DIR, 'runs', 'detect')
            if not os.path.isdir(runs_root):
                continue
            synced = []
            for stage in os.listdir(runs_root):
                w_dir = os.path.join(runs_root, stage, 'weights')
                for fname in ['last.pt', 'best.pt']:
                    src = os.path.join(w_dir, fname)
                    if os.path.exists(src):
                        dst = os.path.join(DRIVE_RESULTS, f'{stage}_{fname}')
                        shutil.copy2(src, dst)
                        synced.append(f'{stage}/{fname}')
            if synced:
                write_status('background-sync', '?', f'synced: {", ".join(synced)}')
                print(f'  🕐 [bg-backup] {time.strftime("%H:%M")} synced {len(synced)} files')
        except Exception as e:
            print(f'  ⚠️  bg-backup error (non-fatal): {e}')

_stop_bg.set()
time.sleep(0.2)
_stop_bg = threading.Event()
_bg_thread = threading.Thread(target=_bg_backup_loop, daemon=True)
_bg_thread.start()
print('✅ Background backup thread started (every 10 min)')

def backup_stage(stage_name):
    run_dir = os.path.join(WORK_DIR, 'runs', 'detect', stage_name)
    saved = []
    for fname in ['best.pt', 'last.pt']:
        src = os.path.join(run_dir, 'weights', fname)
        if os.path.exists(src):
            dst_specific = os.path.join(DRIVE_RESULTS, f'{stage_name}_{fname}')
            shutil.copy2(src, dst_specific)
            if fname == 'best.pt':
                shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_best.pt'))
            if fname == 'last.pt':
                shutil.copy2(src, os.path.join(DRIVE_RESULTS, 'LATEST_last.pt'))
            saved.append(fname)
    write_status(stage_name, 'COMPLETE', f'stage-end backup: {saved}')
    print(f'  💾 Stage backup → Drive: {saved}')

print('\n✅ Cell 2b complete — backup system ACTIVE')
print(f'   Drive backup folder : {DRIVE_RESULTS}')
print(f'   Status file         : {STATUS_FILE}')
print('\n🚦 You can now run Cells 3-9 to train, OR Cell R to resume.')


✅ Epoch callback ready (every 5 epochs)
✅ Background backup thread started (every 10 min)

✅ Cell 2b complete — backup system ACTIVE
   Drive backup folder : /content/drive/MyDrive/maize_yolo26_results_confirm_final
   Status file         : /content/drive/MyDrive/maize_yolo26_results_confirm_final/CHECKPOINT_STATUS.txt

🚦 You can now run Cells 3-9 to train, OR Cell R to resume.


---
## 🔍 Cell 3 — Auto-Detect Dataset Paths & Validate


In [5]:
import glob, os

def find_dir(root, *target_parts):
    target = os.path.join(*target_parts)
    for dirpath, dirnames, _ in os.walk(root):
        candidate = os.path.join(dirpath, target)
        if os.path.isdir(candidate):
            return candidate
    return None

def count_files(d, patterns):
    if not d or not os.path.isdir(d): return 0
    return len(set(f for p in patterns for f in glob.glob(os.path.join(d, p))))

IMG_PATS = ['*.jpg','*.jpeg','*.png','*.JPG','*.JPEG','*.PNG']

UAV_DET_IMG_TRAIN = find_dir(UAV_EXTRACT,'MTDC-UAV', 'Detection', 'images', 'train')
UAV_DET_IMG_TEST  = find_dir(UAV_EXTRACT,'MTDC-UAV', 'Detection', 'images', 'test')
UAV_DET_LBL_TRAIN = find_dir(UAV_EXTRACT,'MTDC-UAV', 'Detection', 'labels', 'train')
UAV_DET_LBL_TEST  = find_dir(UAV_EXTRACT,'MTDC-UAV', 'Detection', 'labels', 'test')
MCD_TRAINVAL_IMG  = find_dir(MCD_EXTRACT, 'trainval', 'images')
MCD_TRAINVAL_LBL  = find_dir(MCD_EXTRACT, 'trainval', 'labels')
MCD_TEST_IMG      = find_dir(MCD_EXTRACT, 'test', 'images')
MCD_TEST_LBL      = find_dir(MCD_EXTRACT, 'test', 'labels')

print('── Path Validation ───────────────────────────────────────────')
checks = [
    ('UAV det images train', UAV_DET_IMG_TRAIN, IMG_PATS),
    ('UAV det images test',  UAV_DET_IMG_TEST,  IMG_PATS),
    ('UAV det labels train', UAV_DET_LBL_TRAIN, ['*.xml']),
    ('UAV det labels test',  UAV_DET_LBL_TEST,  ['*.xml']),
    ('MCD trainval images',  MCD_TRAINVAL_IMG,   IMG_PATS),
    ('MCD trainval labels',  MCD_TRAINVAL_LBL,   ['*.mat']),
    ('MCD test images',      MCD_TEST_IMG,        IMG_PATS),
    ('MCD test labels',      MCD_TEST_LBL,        ['*.mat']),
]
all_ok = True
for name, path, pats in checks:
    n = count_files(path, pats)
    ok = path is not None and n > 0
    if not ok: all_ok = False
    print(f'  {"✅" if ok else "❌"}  {name:<28}  {n:>4} files  →  {path}')

if all_ok:
    print('\n✅ All paths found!')
else:
    print('\n⚠️  Some paths missing. Check Cell 2 extraction output.')
    if UAV_DET_LBL_TRAIN is None and UAV_DET_IMG_TRAIN is not None:
        print('\n⚠️  HINT: Some MTDC versions store .xml alongside images, not in labels/')

print(f'\n── Counts ────────────────────────────────────────────────────')
print(f'  MCD trainval : {count_files(MCD_TRAINVAL_IMG, IMG_PATS)} images')
print(f'  MCD test     : {count_files(MCD_TEST_IMG, IMG_PATS)} images (LOCKED)')
print(f'  UAV train    : {count_files(UAV_DET_IMG_TRAIN, IMG_PATS)} images')
print(f'  UAV test     : {count_files(UAV_DET_IMG_TEST, IMG_PATS)} images')


── Path Validation ───────────────────────────────────────────
  ✅  UAV det images train           500 files  →  /content/dataset/uav/MTDC-UAV/Detection/images/train
  ✅  UAV det images test            300 files  →  /content/dataset/uav/MTDC-UAV/Detection/images/test
  ✅  UAV det labels train           500 files  →  /content/dataset/uav/MTDC-UAV/Detection/labels/train
  ✅  UAV det labels test            300 files  →  /content/dataset/uav/MTDC-UAV/Detection/labels/test
  ✅  MCD trainval images            186 files  →  /content/dataset/mcd/maize_counting_dataset/trainval/images
  ✅  MCD trainval labels            186 files  →  /content/dataset/mcd/maize_counting_dataset/trainval/labels
  ✅  MCD test images                175 files  →  /content/dataset/mcd/maize_counting_dataset/test/images
  ✅  MCD test labels                175 files  →  /content/dataset/mcd/maize_counting_dataset/test/labels

✅ All paths found!

── Counts ────────────────────────────────────────────────────
  MCD train

---
## 🔄 Cell 4 — Parse Annotations

### MCD format
MCD .mat files store REAL bounding boxes in `annotation['bndbox']` as an
array of rows: [x_min, y_min, width, height] (pixel coordinates).
These real boxes are used directly — no synthetic pseudo-boxes.

### UAV format
MTDC-UAV uses Pascal VOC XML with `<bndbox>` (xmin, ymin, xmax, ymax).


In [6]:
import os, glob, random, shutil
import xml.etree.ElementTree as ET
import numpy as np
import scipy.io as sio
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


def load_mcd_real_boxes(mat_path, img_w, img_h):
    """
    Parse REAL bounding boxes from an MCD .mat annotation file.

    The MCD dataset stores ground-truth boxes in:
        annotation['bndbox']  shape: (N, 4)  columns: [x_min, y_min, width, height]

    All coordinates are in absolute pixels.  We convert to YOLO normalised
    cx, cy, nw, nh format.

    Returns a list of [class_id, cx, cy, nw, nh] rows (class_id always 0).
    Raises ValueError with a descriptive message if the expected structure
    is not found, so the caller can log and skip the file.
    """
    rows = []
    mat = sio.loadmat(mat_path)

    ann = mat.get('annotation', None)
    if ann is None:
        raise ValueError(f"No 'annotation' key in mat file. Available keys: {[k for k in mat if not k.startswith('_')]}")

    # annotation is a MATLAB struct stored as a numpy void/object array.
    # Access the bndbox field:
    try:
        bndbox_raw = ann['bndbox']
        # Unwrap nested numpy object arrays produced by scipy.io.loadmat
        # Typical shape after loadmat: (1,1) object wrapping (N,4) array
        while hasattr(bndbox_raw, 'shape') and bndbox_raw.dtype == object:
            if bndbox_raw.shape == ():
                bndbox_raw = bndbox_raw.item()
                break
            bndbox_raw = bndbox_raw.flat[0]
        bndbox = np.array(bndbox_raw, dtype=np.float32)
    except Exception as e:
        raise ValueError(f"Could not access annotation['bndbox']: {e}")

    if bndbox.ndim == 1:
        if bndbox.size == 0:
            # annotation['bndbox'] = [] — legitimately empty image, no boxes
            return []
        # Single box stored as flat vector — reshape to (1, N)
        bndbox = bndbox.reshape(1, -1)

    if bndbox.ndim == 2 and bndbox.shape[0] == 0:
        # Shape (0, N) — empty annotation array, no boxes
        return []

    if bndbox.ndim != 2 or bndbox.shape[1] < 4:
        raise ValueError(f"Unexpected bndbox shape {bndbox.shape}; expected (N, >=4)")

    for row in bndbox:
        x_min, y_min, bw, bh = float(row[0]), float(row[1]), float(row[2]), float(row[3])

        # Clamp to image boundary
        x_min = max(0.0, min(x_min, img_w))
        y_min = max(0.0, min(y_min, img_h))
        bw    = min(bw, img_w - x_min)
        bh    = min(bh, img_h - y_min)

        if bw <= 0 or bh <= 0:
            continue  # degenerate box — skip silently

        cx = float(np.clip((x_min + bw / 2) / img_w, 0.0, 1.0))
        cy = float(np.clip((y_min + bh / 2) / img_h, 0.0, 1.0))
        nw = float(np.clip(bw / img_w, 1e-4, 1.0))
        nh = float(np.clip(bh / img_h, 1e-4, 1.0))
        rows.append([0, cx, cy, nw, nh])

    return rows


def load_mcd_boxes_safe(mat_path, img_w, img_h):
    """
    Wrapper around load_mcd_real_boxes that catches all errors and returns
    an empty list on failure (with a warning print).
    Empty annotations (bndbox=[]) are valid — no warning is printed for those.
    """
    try:
        return load_mcd_real_boxes(mat_path, img_w, img_h)
    except Exception as e:
        print(f'  [WARN mat] {Path(mat_path).name}: {e}')
        return []


def load_voc_xml(xml_path, img_w, img_h):
    """Parse Pascal VOC XML annotation → YOLO normalised format."""
    rows = []
    try:
        root = ET.parse(xml_path).getroot()
        size = root.find('size')
        if size is not None:
            try:
                w, h = int(size.find('width').text), int(size.find('height').text)
                if w > 0 and h > 0: img_w, img_h = w, h
            except Exception:
                pass
        for obj in root.findall('object'):
            bb = obj.find('bndbox')
            if bb is None: continue
            try:
                xmin = float(bb.find('xmin').text); ymin = float(bb.find('ymin').text)
                xmax = float(bb.find('xmax').text); ymax = float(bb.find('ymax').text)
            except Exception as e:
                print(f'  [WARN xml coord] {Path(xml_path).name}: {e}'); continue
            xmin = max(0., min(xmin, img_w)); ymin = max(0., min(ymin, img_h))
            xmax = max(0., min(xmax, img_w)); ymax = max(0., min(ymax, img_h))
            bw = xmax - xmin; bh = ymax - ymin
            if bw <= 0 or bh <= 0: continue
            cx = float(np.clip((xmin + bw/2) / img_w, 0, 1))
            cy = float(np.clip((ymin + bh/2) / img_h, 0, 1))
            nw = float(np.clip(bw / img_w, 1e-4, 1))
            nh = float(np.clip(bh / img_h, 1e-4, 1))
            rows.append([0, cx, cy, nw, nh])
    except Exception as e:
        print(f'  [WARN xml] {Path(xml_path).name}: {e}')
    return rows


def get_img_size(img_path):
    try:
        with Image.open(img_path) as im: im.verify()
        with Image.open(img_path) as im: return im.size  # (width, height)
    except Exception as e:
        print(f'  [WARN corrupt] {Path(img_path).name}: {e}')
        return None


def collect_pairs(img_dir, lbl_dir, lbl_ext):
    if not img_dir or not lbl_dir: return []
    imgs = sorted(set(f for p in IMG_PATS for f in glob.glob(os.path.join(img_dir, p))))
    pairs, missing = [], 0
    for img_p in imgs:
        lbl_p = os.path.join(lbl_dir, Path(img_p).stem + lbl_ext)
        if os.path.exists(lbl_p): pairs.append((img_p, lbl_p))
        else: missing += 1
    if missing: print(f'    ⚠️  {missing} images had no matching label — skipped')
    return pairs


print('Collecting pairs…')
# ── MCD: combine official trainval + test, then stratified re-split ───────
# Stratify by annotation density (empty vs non-empty) so each split gets
# a proportional share of background images — plain shuffle can cluster
# all empty annotations into one split by chance.
mcd_all = collect_pairs(MCD_TRAINVAL_IMG, MCD_TRAINVAL_LBL, '.mat') \
        + collect_pairs(MCD_TEST_IMG,     MCD_TEST_LBL,     '.mat')

def is_empty_annotation(mat_path):
    """Return True if this .mat file has zero valid bounding boxes."""
    try:
        mat  = sio.loadmat(mat_path)
        ann  = mat.get('annotation', None)
        if ann is None:
            return True
        raw = ann['bndbox']
        while hasattr(raw, 'shape') and raw.dtype == object:
            if raw.shape == (): raw = raw.item(); break
            raw = raw.flat[0]
        arr = np.array(raw, dtype=np.float32)
        if arr.ndim == 1:
            return arr.size == 0
        return arr.shape[0] == 0
    except Exception:
        return True   # unreadable → treat as empty

random.seed(SEED)
empty_pairs    = [(i, l) for i, l in mcd_all if     is_empty_annotation(l)]
nonempty_pairs = [(i, l) for i, l in mcd_all if not is_empty_annotation(l)]
random.shuffle(empty_pairs)
random.shuffle(nonempty_pairs)

print(f'  MCD annotation breakdown — '
      f'non-empty: {len(nonempty_pairs)}  empty: {len(empty_pairs)}  '
      f'({100*len(empty_pairs)/max(1,len(mcd_all)):.1f}% background)')

def stratified_split(a, b, r_test, r_val):
    """Split two buckets proportionally and concatenate matching slices."""
    def _split(lst):
        n      = len(lst)
        n_test = max(0, round(n * r_test))
        n_val  = max(0, round(n * r_val))
        return lst[:n_test], lst[n_test:n_test+n_val], lst[n_test+n_val:]
    a_te, a_va, a_tr = _split(a)
    b_te, b_va, b_tr = _split(b)
    return a_tr + b_tr, a_va + b_va, a_te + b_te

mcd_tr, mcd_val, mcd_test = stratified_split(
    empty_pairs, nonempty_pairs, r_test=0.15, r_val=0.15
)

# Shuffle assembled splits so empty/non-empty aren't block-ordered
random.shuffle(mcd_tr)
random.shuffle(mcd_val)
random.shuffle(mcd_test)

# UAV splits unchanged (UAV test IS a locked benchmark)
uav_train = collect_pairs(UAV_DET_IMG_TRAIN, UAV_DET_LBL_TRAIN, '.xml')
uav_test  = collect_pairs(UAV_DET_IMG_TEST,  UAV_DET_LBL_TEST,  '.xml')
random.shuffle(uav_train)
n_uv    = max(1, int(len(uav_train) * 0.20))
uav_val = uav_train[:n_uv]
uav_tr  = uav_train[n_uv:]

print(f'\n── Splits ──────────────────────────────────────────────────')
for name, split in [('train', mcd_tr), ('val', mcd_val), ('test', mcd_test)]:
    n_e = sum(1 for _, l in split if is_empty_annotation(l))
    print(f'  MCD  {name:<6}: {len(split):>3} total  '
          f'({len(split)-n_e} with boxes  +  {n_e} empty)')
print(f'  UAV  train:{len(uav_tr)}  val:{len(uav_val)}  test:{len(uav_test)} (locked)')
print(f'  Combined train:{len(mcd_tr)+len(uav_tr)}  val:{len(mcd_val)+len(uav_val)}')

# ── Sanity check: verify real MCD boxes are parsed correctly ─────────────
print('\nSanity check — 3 MCD samples (using REAL bndbox):')
for img_p, lbl_p in mcd_tr[:3]:
    sz = get_img_size(img_p)
    if sz:
        boxes = load_mcd_boxes_safe(lbl_p, sz[0], sz[1])
        print(f'  {Path(img_p).name:<40} {sz} → {len(boxes)} real boxes')
print('Sanity check — 3 UAV samples:')
for img_p, lbl_p in uav_tr[:3]:
    sz = get_img_size(img_p)
    if sz: print(f'  {Path(img_p).name:<40} {sz} → {len(load_voc_xml(lbl_p,sz[0],sz[1]))} boxes')


  MCD annotation breakdown — non-empty: 344  empty: 17  (4.7% background)

── Splits ──────────────────────────────────────────────────
  MCD  train : 251 total  (240 with boxes  +  11 empty)
  MCD  val   :  55 total  (52 with boxes  +  3 empty)
  MCD  test  :  55 total  (52 with boxes  +  3 empty)
  UAV  train:400  val:100  test:300 (locked)
  Combined train:651  val:155

Sanity check — 3 MCD samples (using REAL bndbox):
  T0002_XM_20110729160224_01.jpg           (3648, 2736) → 6 real boxes
  T0001_XM_20120808110251_01.jpg           (3648, 2736) → 50 real boxes
  T0001_XM_20120802140257_02.jpg           (3648, 2736) → 4 real boxes
Sanity check — 3 UAV samples:
  DJI_0548 (2)_0_0.JPG                     (2736, 1824) → 32 boxes
  DJI_0018 (2)_0_1.JPG                     (2736, 1824) → 77 boxes
  DJI_0103 (2)_1_1.JPG                     (2736, 1824) → 48 boxes


---
## ✂️ Cell 5 — SAHI Tiling for UAV Images


In [7]:
TILE_SIZE    = 640
TILE_OVERLAP = 0.40  # 0.40 reduces border-cut misses for dense small tassels (vs default 0.30)
MIN_VIS_FRAC = 0.30

def tile_uav_image(img_path, lbl_path, out_img_dir, out_lbl_dir):
    try:
        img = np.array(Image.open(img_path).convert('RGB'))
    except Exception as e:
        print(f'  [WARN tile] {Path(img_path).name}: {e}'); return 0
    H, W  = img.shape[:2]
    stem  = Path(img_path).stem
    stride = int(TILE_SIZE * (1 - TILE_OVERLAP))
    raw = load_voc_xml(lbl_path, W, H)
    if not raw: return 0
    px = [[(cx-nw/2)*W,(cy-nh/2)*H,(cx+nw/2)*W,(cy+nh/2)*H] for _,cx,cy,nw,nh in raw]

    saved, idx = 0, 0
    for y0 in range(0, H, stride):
        for x0 in range(0, W, stride):
            x2t=min(x0+TILE_SIZE,W); y2t=min(y0+TILE_SIZE,H)
            tw=x2t-x0; th=y2t-y0
            if tw < TILE_SIZE//4 or th < TILE_SIZE//4: continue
            tile_boxes = []
            for box in px:
                ix1=max(box[0],x0); iy1=max(box[1],y0)
                ix2=min(box[2],x2t); iy2=min(box[3],y2t)
                if ix2<=ix1 or iy2<=iy1: continue
                inter=(ix2-ix1)*(iy2-iy1)
                ba=(box[2]-box[0])*(box[3]-box[1])
                if ba<=0 or inter/ba<MIN_VIS_FRAC: continue
                cx_t=float(np.clip(((ix1+ix2)/2-x0)/tw,0,1))
                cy_t=float(np.clip(((iy1+iy2)/2-y0)/th,0,1))
                nw_t=float(np.clip((ix2-ix1)/tw,1e-4,1))
                nh_t=float(np.clip((iy2-iy1)/th,1e-4,1))
                tile_boxes.append([0,cx_t,cy_t,nw_t,nh_t])
            if tile_boxes:
                tname = f'{stem}_t{idx:04d}_U'
                tile_arr = img[y0:y2t, x0:x2t]
                if tile_arr.shape[0]!=TILE_SIZE or tile_arr.shape[1]!=TILE_SIZE:
                    padded=np.zeros((TILE_SIZE,TILE_SIZE,3),dtype=np.uint8)
                    padded[:tile_arr.shape[0],:tile_arr.shape[1]]=tile_arr
                    tile_arr=padded
                Image.fromarray(tile_arr).save(os.path.join(out_img_dir,tname+'.jpg'),quality=95)
                with open(os.path.join(out_lbl_dir,tname+'.txt'),'w') as f:
                    for b in tile_boxes:
                        f.write(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')
                # Store exact pixel offset so merge step never needs to guess grid shape
                with open(os.path.join(out_lbl_dir,tname+'.offset'),'w') as f:
                    f.write(f'{x0} {y0}\n')
                saved+=1
            idx+=1
    return saved

# ── Create output dirs for all three splits ───────────────────────────────
for split_name in ('train', 'val', 'test'):
    os.makedirs(os.path.join(TILED_DIR, split_name, 'images'), exist_ok=True)
    os.makedirs(os.path.join(TILED_DIR, split_name, 'labels'), exist_ok=True)

totals = {}
# ⚠️  test split MUST be tiled with identical TILE_SIZE/OVERLAP so the model
#     sees the same scale at eval as it saw during training.
for split_name, pairs in [('train', uav_tr), ('val', uav_val), ('test', uav_test)]:
    print(f'Tiling UAV {split_name} ({len(pairs)} images)…')
    out_i = os.path.join(TILED_DIR, split_name, 'images')
    out_l = os.path.join(TILED_DIR, split_name, 'labels')
    total = 0
    for img_p, lbl_p in tqdm(pairs, unit='img'):
        total += tile_uav_image(img_p, lbl_p, out_i, out_l)
    totals[split_name] = total
    print(f'  → {total} tiles saved')
print(f'\n✅ SAHI tiling complete  train:{totals["train"]}  val:{totals["val"]}  test:{totals["test"]}')


Tiling UAV train (400 images)…


  0%|          | 0/400 [00:00<?, ?img/s]

  → 10668 tiles saved
Tiling UAV val (100 images)…


  0%|          | 0/100 [00:00<?, ?img/s]

  → 2671 tiles saved
Tiling UAV test (300 images)…


  0%|          | 0/300 [00:00<?, ?img/s]

  → 8643 tiles saved

✅ SAHI tiling complete  train:10668  val:2671  test:8643


---
## 📦 Cell 6 — Build Final YOLO26 Dataset


In [8]:
import os, glob, shutil
from PIL import Image
from tqdm.notebook import tqdm

for s in ('train','val','test'):
    os.makedirs(os.path.join(DATASET_DIR,'images',s),exist_ok=True)
    os.makedirs(os.path.join(DATASET_DIR,'labels',s),exist_ok=True)

def write_yolo_label(dst_path, boxes):
    with open(dst_path,'w') as f:
        for b in boxes:
            f.write(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')

def add_mcd_pair(img_p, lbl_p, split, tag='G'):
    """Copy an MCD image+label pair into the YOLO dataset using REAL bndbox.
    Images with zero annotations are kept — an empty .txt label is written.
    YOLO treats an empty label file as a valid background/negative sample.
    """
    stem=Path(img_p).stem; ext=Path(img_p).suffix
    new_stem=f'{stem}_{tag}' if not stem.endswith(f'_{tag}') else stem
    dst_img=os.path.join(DATASET_DIR,'images',split,new_stem+ext)
    dst_lbl=os.path.join(DATASET_DIR,'labels',split,new_stem+'.txt')
    if os.path.exists(dst_img) and os.path.exists(dst_lbl): return True
    size=get_img_size(img_p)
    if size is None: return False
    # ── Use real bounding boxes from annotation['bndbox'] ─────────────────
    boxes = load_mcd_boxes_safe(lbl_p, size[0], size[1])
    # boxes == [] is valid (empty/background image) — write an empty label file
    # so YOLO sees it as a negative sample rather than skipping it entirely.
    shutil.copy2(img_p, dst_img)
    write_yolo_label(dst_lbl, boxes)   # writes empty file when boxes == []
    return True

print('Building TRAIN…')
for img_p,lbl_p in tqdm(mcd_tr,desc='  MCD train'): add_mcd_pair(img_p,lbl_p,'train','G')
for img_p in tqdm(sorted(set(glob.glob(os.path.join(TILED_DIR,'train','images','*.jpg')))),desc='  UAV tiles'):
    stem=Path(img_p).stem
    lp=os.path.join(TILED_DIR,'train','labels',stem+'.txt')
    op=os.path.join(TILED_DIR,'train','labels',stem+'.offset')
    di=os.path.join(DATASET_DIR,'images','train',os.path.basename(img_p))
    dl=os.path.join(DATASET_DIR,'labels','train',stem+'.txt')
    do=os.path.join(DATASET_DIR,'labels','train',stem+'.offset')
    if not os.path.exists(di): shutil.copy2(img_p,di)
    if os.path.exists(lp) and not os.path.exists(dl): shutil.copy2(lp,dl)
    if os.path.exists(op) and not os.path.exists(do): shutil.copy2(op,do)

print('Building VAL…')
for img_p,lbl_p in tqdm(mcd_val,desc='  MCD val'): add_mcd_pair(img_p,lbl_p,'val','G')
for img_p in tqdm(sorted(set(glob.glob(os.path.join(TILED_DIR,'val','images','*.jpg')))),desc='  UAV val tiles'):
    stem=Path(img_p).stem
    lp=os.path.join(TILED_DIR,'val','labels',stem+'.txt')
    op=os.path.join(TILED_DIR,'val','labels',stem+'.offset')
    di=os.path.join(DATASET_DIR,'images','val',os.path.basename(img_p))
    dl=os.path.join(DATASET_DIR,'labels','val',stem+'.txt')
    do=os.path.join(DATASET_DIR,'labels','val',stem+'.offset')
    if not os.path.exists(di): shutil.copy2(img_p,di)
    if os.path.exists(lp) and not os.path.exists(dl): shutil.copy2(lp,dl)
    if os.path.exists(op) and not os.path.exists(do): shutil.copy2(op,do)

print('Building TEST…')
for img_p,lbl_p in tqdm(mcd_test,desc='  MCD test'): add_mcd_pair(img_p,lbl_p,'test','G')

# UAV test — use the same 640×640 tiles produced in Cell 5.
# ⚠️  Do NOT copy full-resolution UAV images here: the model was trained on
#     640px tiles and would see a completely different scale at test time.
print('Adding UAV test tiles (pre-tiled in Cell 5)…')
uav_test_tile_imgs = sorted(glob.glob(os.path.join(TILED_DIR,'test','images','*.jpg')))
for img_p in tqdm(uav_test_tile_imgs, desc='  UAV test tiles'):
    stem = Path(img_p).stem
    lp   = os.path.join(TILED_DIR, 'test', 'labels', stem + '.txt')
    op   = os.path.join(TILED_DIR, 'test', 'labels', stem + '.offset')
    di   = os.path.join(DATASET_DIR, 'images', 'test', os.path.basename(img_p))
    dl   = os.path.join(DATASET_DIR, 'labels', 'test', stem + '.txt')
    do   = os.path.join(DATASET_DIR, 'labels', 'test', stem + '.offset')
    if not os.path.exists(di): shutil.copy2(img_p, di)
    if os.path.exists(lp) and not os.path.exists(dl): shutil.copy2(lp, dl)
    if os.path.exists(op) and not os.path.exists(do): shutil.copy2(op, do)
if not uav_test_tile_imgs:
    print('  ⚠️  No UAV test tiles found — re-run Cell 5 first')

# Write combined YAML
with open(YAML_PATH,'w') as f:
    f.write(f'path : {os.path.abspath(DATASET_DIR)}\n')
    f.write('train: images/train\nval  : images/val\ntest : images/test\n\n')
    f.write("nc   : 1\nnames: ['tassel']\n")

# Ground-only dataset (Stage 1)
for s in ('train','val'):
    os.makedirs(os.path.join(GROUND_DIR,'images',s),exist_ok=True)
    os.makedirs(os.path.join(GROUND_DIR,'labels',s),exist_ok=True)
for s in ('train','val'):
    for img_p in glob.glob(os.path.join(DATASET_DIR,'images',s,'*_G.*')):
        stem=Path(img_p).stem; ext=Path(img_p).suffix
        di=os.path.join(GROUND_DIR,'images',s,os.path.basename(img_p))
        dl=os.path.join(GROUND_DIR,'labels',s,stem+'.txt')
        sl=os.path.join(DATASET_DIR,'labels',s,stem+'.txt')
        if not os.path.exists(di): shutil.copy2(img_p,di)
        if os.path.exists(sl) and not os.path.exists(dl): shutil.copy2(sl,dl)
with open(GROUND_YAML,'w') as f:
    f.write(f'path : {os.path.abspath(GROUND_DIR)}\n')
    f.write('train: images/train\nval  : images/val\n\n')
    f.write("nc   : 1\nnames: ['tassel']\n")

def count_split(base,s):
    return len(set(f for p in IMG_PATS for f in glob.glob(os.path.join(base,'images',s,p))))

print(f'\n✅ DATASET READY')
print(f'   train  : {count_split(DATASET_DIR,"train")}')
print(f'   val    : {count_split(DATASET_DIR,"val")}')
print(f'   test   : {count_split(DATASET_DIR,"test")}')
print(f'   ground train:{count_split(GROUND_DIR,"train")} val:{count_split(GROUND_DIR,"val")}')

shutil.copy2(YAML_PATH, os.path.join(DRIVE_RESULTS,'dataset.yaml'))
shutil.copy2(GROUND_YAML, os.path.join(DRIVE_RESULTS,'dataset_ground.yaml'))
print('   YAMLs backed up to Drive ✅')


Building TRAIN…


  MCD train:   0%|          | 0/251 [00:00<?, ?it/s]

  UAV tiles:   0%|          | 0/10668 [00:00<?, ?it/s]

Building VAL…


  MCD val:   0%|          | 0/55 [00:00<?, ?it/s]

...

Building TEST…


  MCD test:   0%|          | 0/55 [00:00<?, ?it/s]

Adding UAV test tiles (pre-tiled in Cell 5)…


  UAV test tiles:   0%|          | 0/8643 [00:00<?, ?it/s]


✅ DATASET READY
   train  : 10919
   val    : 2726
   test   : 8698
   ground train:251 val:55
   YAMLs backed up to Drive ✅


---
## 🚀 Cell 7 — Stage 1: Backbone Frozen, Ground-Only (50 epochs)
Warmup: trains only the detection head on ground images, backbone locked.

Note: albumentations is NOT used here. YOLO's built-in HSV/flip/mosaic
augmentations are applied via the training arguments below, which is the
correct and supported way to augment with ultralytics.


In [ ]:
from ultralytics import YOLO
import torch

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device:{DEVICE}  Model:{MODEL_SIZE}  Batch:{BATCH}  ImgSz:{IMG_SIZE}')

write_status(S1_NAME, 0, 'starting Stage 1')
print(f'\n── STAGE 1: backbone frozen, ground-only, 50 epochs ──')
model_s1 = YOLO(f'{MODEL_SIZE}.pt')
try:
    model_s1.add_callback("on_train_epoch_end", _BACKUP_CALLBACKS["on_train_epoch_end"])
    print("✅ Backup callback attached")
except Exception as _cb_e:
    print(f"⚠️  Callback attach failed (non-fatal): {_cb_e}")
model_s1.train(
    data=GROUND_YAML, epochs=50, imgsz=IMG_SIZE, batch=BATCH,
    device=DEVICE, project=f'{WORK_DIR}/runs/detect', name=S1_NAME,
    freeze=5,               # freeze only early stem/C2f layers (0-4); deeper backbone
                            # adapts to agricultural imagery — COCO→crop is a large domain gap
    optimizer='auto',       # auto → AdamW for <10k iters, MuSGD for longer (YOLO26 native)
    lr0=0.0005,             # conservative frozen-head warmup; backbone not updating
    lrf=0.15,               # gentle LR decay for short warmup stage
    momentum=0.948,         # YOLO26s official recipe value
    weight_decay=0.00027,   # YOLO26s official recipe value
    warmup_epochs=5, warmup_momentum=0.8,
    close_mosaic=10,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.3, fliplr=0.5, degrees=10.0,
    translate=0.1, scale=0.5, mosaic=1.0,
    mixup=0.0, copy_paste=0.0,  # keep off in frozen warmup — head only
    iou=TRAIN_IOU,   # ← STAL label-assignment IoU threshold
    patience=999, save=True, plots=True, exist_ok=True,
    cache=False,   # prevent stale .cache if dataset is rebuilt between runs
    verbose=False, workers=2,
)
print(f'✅ Stage 1 complete')
backup_stage(S1_NAME)


Device:0  Model:yolo26s  Batch:16  ImgSz:640

── STAGE 1: backbone frozen, ground-only, 50 epochs ──
✅ Backup callback attached
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/maize_work/dataset_ground_only/dataset.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=5, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.15, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momen

---
## 🚀 Cell 8 — Stage 2: All Layers, Combined Dataset (80 epochs)

Note: albumentations is NOT used here. Augmentation is handled entirely
by YOLO's built-in pipeline via the parameters passed to model.train().


In [ ]:
from ultralytics import YOLO

write_status(S2_NAME, 0, 'starting Stage 2')
print(f'\n── STAGE 2: all layers, combined data, 80 epochs ──')
model_s2 = YOLO(S1_BEST)
try:
    model_s2.add_callback("on_train_epoch_end", _BACKUP_CALLBACKS["on_train_epoch_end"])
    print("✅ Backup callback attached")
except Exception as _cb_e:
    print(f"⚠️  Callback attach failed (non-fatal): {_cb_e}")
model_s2.train(
    data=YAML_PATH, epochs=80, imgsz=IMG_SIZE, batch=BATCH,
    device=DEVICE, project=f'{WORK_DIR}/runs/detect', name=S2_NAME,
    freeze=0,
    optimizer='auto',       # auto → MuSGD for 80 epochs on combined dataset (YOLO26 native)
    lr0=0.0008,             # scaled from official 0.00038 @ batch=128 → batch=16 (~8x smaller)
                            # linear LR scaling rule + small-dataset upward correction
    lrf=0.1,                # gentle decay; small dataset benefits from not collapsing LR too fast
    momentum=0.948,         # YOLO26s official recipe value
    weight_decay=0.00027,   # YOLO26s official recipe value (was 0.0005, over-regularising)
    warmup_epochs=2, warmup_momentum=0.8,
    close_mosaic=10,
    box=9.83, cls=0.65,     # YOLO26s official loss weights
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.3, fliplr=0.5, degrees=15.0,
    translate=0.1,
    scale=0.9,              # YOLO26s official: 0.9 handles multi-altitude UAV scale variance
    mosaic=1.0,
    mixup=0.05,             # YOLO26s official: light mixup improves generalisation
    copy_paste=0.1,         # light copy-paste only: high values inflate tassel density unrealistically
                            # in already-dense scenes, confusing the one-to-one head (research: Ghiasi et al.)
    iou=TRAIN_IOU,          # ← STAL label-assignment IoU threshold
    patience=30,            # early stopping: small dataset converges fast; 30 epochs no-improve = stop
    save=True, save_period=10,
    cache=False,   # prevent stale .cache if dataset is rebuilt between runs
    plots=True, exist_ok=True, verbose=True, workers=2,
)
print(f'✅ Stage 2 complete')
backup_stage(S2_NAME)



── STAGE 2: all layers, combined data, 80 epochs ──
✅ Backup callback attached
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=9.83, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.65, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/maize_work/dataset/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0008, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content/maize_work/runs/detect/maize_yolo26_s1/weights/best.pt, momentum=0

---
## 🚀 Cell 9 — Stage 3: Fine-Tune at 1280px (80 epochs)

Note: albumentations is NOT used here. Augmentation is handled entirely
by YOLO's built-in pipeline via the parameters passed to model.train().


In [ ]:
from ultralytics import YOLO

write_status(S3_NAME, 0, 'starting Stage 3')
print(f'\n── STAGE 3: fine-tune, 80 epochs, imgsz=1280, lr=0.0001 ──')

# ── Safe batch size for Stage 3 (1280px is VRAM-heavy) ────────────────────
# batch=-1 triggers ultralytics auto-batch: binary-searches the largest batch
# that fits in GPU memory with a 60% VRAM budget.  Falls back to batch=2 if
# auto-batch itself OOMs, which is safe on any T4/V100.
import torch as _torch
_vram_gb = _torch.cuda.get_device_properties(0).total_memory / 1e9 if _torch.cuda.is_available() else 0
if _vram_gb >= 20:
    S3_BATCH = 8   # A100 / V100 32GB — can afford more
elif _vram_gb >= 14:
    S3_BATCH = -1  # T4 15GB — let ultralytics auto-batch find the safe size
else:
    S3_BATCH = 2   # smaller GPU — conservative safe minimum
print(f'  Stage 3 batch: {S3_BATCH}  (GPU VRAM: {_vram_gb:.1f} GB)')

model_s3 = YOLO(S2_BEST)
try:
    model_s3.add_callback("on_train_epoch_end", _BACKUP_CALLBACKS["on_train_epoch_end"])
    print("✅ Backup callback attached")
except Exception as _cb_e:
    print(f"⚠️  Callback attach failed (non-fatal): {_cb_e}")
model_s3.train(
    data=YAML_PATH, epochs=80, imgsz=1280, batch=S3_BATCH,
    device=DEVICE, project=f'{WORK_DIR}/runs/detect', name=S3_NAME,
    freeze=0,
    optimizer='auto',       # auto → MuSGD for fine-tune run (YOLO26 native)
    lr0=0.0001,             # low LR for fine-tune; backbone already converged
    lrf=0.1,                # gentle final decay
    momentum=0.948,         # YOLO26s official recipe value
    weight_decay=0.00027,   # YOLO26s official recipe value
    warmup_epochs=3,
    close_mosaic=10,        # close mosaic last 10 epochs (not 20; 10 is YOLO26 default)
    box=9.83, cls=0.65,     # YOLO26s official loss weights
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    flipud=0.3, fliplr=0.5, degrees=15.0,
    translate=0.1,
    scale=0.9,              # match Stage 2 scale augmentation
    mosaic=0.5,             # reduced mosaic at 1280px — memory-safe, still diverse
    mixup=0.05,             # light mixup as per YOLO26s recipe
    copy_paste=0.1,         # same reasoning as Stage 2 — keep light for fine-tune stage
    iou=TRAIN_IOU,          # ← STAL label-assignment IoU threshold
    patience=20,            # tighter early stopping for fine-tune: overfit risk rises at 1280px
    save=True, save_period=5,
    cache=False,   # prevent stale .cache if dataset is rebuilt between runs
    plots=True, exist_ok=True, verbose=True, workers=2,
)
print(f'\n✅ ALL 3 STAGES COMPLETE!')
print(f'   Best weights: {BEST_WEIGHTS}')
backup_stage(S3_NAME)
write_status(S3_NAME, 'DONE', 'ALL STAGES COMPLETE')



── STAGE 3: fine-tune, 80 epochs, imgsz=1280, lr=0.0001 ──
  Stage 3 batch: -1  (GPU VRAM: 15.6 GB)
✅ Backup callback attached
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=9.83, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.65, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/maize_work/dataset/dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=0, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/content/maize_work/runs/d

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 🔁 Cell R — Resume After Crash / Timeout
**Run this INSTEAD of Cells 7-9 when you reconnect after a crash.**

**Step-by-step after reconnect:**
1. Run **Cell 1** (reinstall)
2. Run **Cell 2** (remount Drive, set paths)
3. Run **Cell 2b** (re-activate backup system)
4. Read `CHECKPOINT_STATUS.txt` in your Drive to see where it crashed
5. Set `RESUME_STAGE` below and run this cell


In [ ]:
import shutil, os, torch
from ultralytics import YOLO

# ✏️ SET THIS to whichever stage crashed: 'S1', 'S2', or 'S3'
RESUME_STAGE = 'S3'

stage_map = {'S1': S1_NAME, 'S2': S2_NAME, 'S3': S3_NAME}
if RESUME_STAGE not in stage_map:
    raise ValueError(f'RESUME_STAGE must be S1, S2, or S3. Got: {RESUME_STAGE}')

stage_name  = stage_map[RESUME_STAGE]
local_w_dir = os.path.join(WORK_DIR, 'runs', 'detect', stage_name, 'weights')
local_last  = os.path.join(local_w_dir, 'last.pt')

candidates = [
    (os.path.join(DRIVE_RESULTS, f'{stage_name}_last.pt'), 'per-stage last.pt'),
    (os.path.join(DRIVE_RESULTS, 'LATEST_last.pt'),         'LATEST_last.pt'),
    (os.path.join(DRIVE_RESULTS, f'{stage_name}_best.pt'), 'per-stage best.pt ⚠️ (may repeat epochs)'),
    (os.path.join(DRIVE_RESULTS, 'LATEST_best.pt'),         'LATEST_best.pt ⚠️ (may repeat epochs)'),
]

print(f'Looking for checkpoint for stage {RESUME_STAGE} ({stage_name})…')
restore_from, restore_label = None, None
for path, label in candidates:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  ✅ Found [{label}]: {path}  ({size_mb:.1f} MB)')
        restore_from, restore_label = path, label
        break
    else:
        print(f'  ❌ Not found: {path}')

if restore_from is None:
    print('\n❌ No checkpoint found in Drive.')
    print(f'   Check your Drive folder: {DRIVE_RESULTS}')
    print('   If Drive is empty, you must restart from Cell 7.')
    raise SystemExit('Nothing to resume from')

os.makedirs(local_w_dir, exist_ok=True)
shutil.copy2(restore_from, local_last)
print(f'\n✅ Restored [{restore_label}] → {local_last}')
print(f'   File size: {os.path.getsize(local_last)/1e6:.1f} MB')

DEVICE = '0' if torch.cuda.is_available() else 'cpu'

STATUS_FILE = os.path.join(DRIVE_RESULTS, 'CHECKPOINT_STATUS.txt')
if os.path.exists(STATUS_FILE):
    print('\n── Last checkpoint status ───────────────────────────────')
    with open(STATUS_FILE) as f:
        print(f.read())

write_status(stage_name, '?', f'RESUMING from {restore_label}')
model = YOLO(local_last)
print(f'\n🔁 Resuming Stage {RESUME_STAGE} from last checkpoint…')
model.train(resume=True)

print(f'\n✅ Stage {RESUME_STAGE} complete!')
backup_stage(stage_name)

next_map = {'S1': 'Cell 8 (Stage 2)', 'S2': 'Cell 9 (Stage 3)', 'S3': 'Cell 10 (Eval)'}
print(f'\n👉 Next step: run {next_map.get(RESUME_STAGE, "Cell 10")}')


Looking for checkpoint for stage S3 (maize_yolo26_s3_final)…
  ✅ Found [per-stage last.pt]: /content/drive/MyDrive/maize_yolo26_results_confirm_final/maize_yolo26_s3_final_last.pt  (60.4 MB)

✅ Restored [per-stage last.pt] → /content/maize_work/runs/detect/maize_yolo26_s3_final/weights/last.pt
   File size: 60.4 MB

── Last checkpoint status ───────────────────────────────
stage=background-sync
epoch=?
note=synced: maize_yolo26_s3_final/last.pt, maize_yolo26_s3_final/best.pt
time=2026-06-05 09:17:24


🔁 Resuming Stage S3 from last checkpoint…
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=9.83, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.65, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/maize_work/dataset/dataset.yaml, degrees=15.0, deterministic=T

---
## 📈 Cell 10 — Training Curves


In [ ]:
from IPython.display import Image as IPImage, display
import os
for sname in [S1_NAME, S2_NAME, S3_NAME]:
    for plot in ['results.png','confusion_matrix.png','PR_curve.png','F1_curve.png']:
        p = os.path.join(WORK_DIR, 'runs', 'detect', sname, plot)
        if os.path.exists(p):
            print(f'── {sname} / {plot} ──')
            display(IPImage(p))


---
## 📊 Cell 11 — Evaluation

### Design note: tile-aware counting for UAV images
UAV test images were tiled (Cell 5) so the model always sees 640×640 crops —
the same resolution it trained on.  This means the test/ folder contains N
tiles per original UAV image, not 1.

**Two separate evaluation modes are used:**

1. **mAP** — computed tile-by-tile via `model.val()`.  Box-level IoU metrics
   are naturally tile-level and are valid as-is.

2. **Counting MAE/RMSE/R²** — must be per *original image*, not per tile.
   `run_counting_merged()` groups tiles by original stem, runs inference on
   each tile, maps detections back to full-image pixel space, applies a final
   cross-tile NMS to remove duplicates in the 30% overlap zones, then sums
   the surviving boxes.  This gives one (gt_count, pred_count) pair per
   original image, which is the correct unit for counting metrics.

   MCD ground images are single-tile (never tiled), so they go through the
   simpler single-image path automatically.


In [ ]:
import numpy as np, glob, os, csv, collections
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

EVAL_CONF  = EVAL_CONF_THR
model_eval = YOLO(BEST_WEIGHTS)


def counting_metrics(gt_l, pred_l):
    gt=np.array(gt_l,dtype=float); pred=np.array(pred_l,dtype=float); diff=pred-gt
    mae=float(np.mean(np.abs(diff))); rmse=float(np.sqrt(np.mean(diff**2)))
    gt_var=float(np.sum((gt-gt.mean())**2))
    r2=float(1-np.sum(diff**2)/(gt_var+1e-9)) if gt_var>1e-9 else float('nan')
    return mae, rmse, r2, gt, pred


def _nms_boxes(boxes_xyxy, scores, iou_thr):
    """Pure-numpy NMS.  boxes: (N,4) xyxy, scores: (N,).  Returns kept indices."""
    if len(boxes_xyxy) == 0:
        return []
    x1,y1,x2,y2 = boxes_xyxy[:,0],boxes_xyxy[:,1],boxes_xyxy[:,2],boxes_xyxy[:,3]
    areas = (x2-x1)*(y2-y1)
    order = scores.argsort()[::-1]
    keep  = []
    while order.size:
        i = order[0]; keep.append(i); order = order[1:]
        xx1=np.maximum(x1[i],x1[order]); yy1=np.maximum(y1[i],y1[order])
        xx2=np.minimum(x2[i],x2[order]); yy2=np.minimum(y2[i],y2[order])
        inter=np.maximum(0,xx2-xx1)*np.maximum(0,yy2-yy1)
        union=areas[i]+areas[order]-inter
        iou=np.where(union>0,inter/union,0)
        order=order[iou<=iou_thr]
    return keep


def _parse_tile_offset(stem):
    """
    Extract (original_stem, tile_col_offset, tile_row_offset) from a UAV tile stem.
    Tile stems look like:  <orig>_t0003_U
    The offset is reconstructed from the tile index and the tiling parameters
    TILE_SIZE / TILE_OVERLAP defined in Cell 5.
    Returns (orig_stem, x0, y0) in pixels, or (stem, 0, 0) if not a tile.
    """
    import re
    m = re.match(r'^(.+)_t(\d{4})_U$', stem)
    if not m:
        return stem, 0, 0
    orig = m.group(1)
    idx  = int(m.group(2))
    stride = int(TILE_SIZE * (1 - TILE_OVERLAP))
    # Tile index is assigned in row-major order inside tile_uav_image().
    # We need the original image width to know how many columns there are,
    # but we don't have that here.  Instead we store x0,y0 inside the tile
    # label file as a comment — but we didn't do that.
    # Workaround: reconstruct from sorted tile index.
    # This is only used to group tiles; offsets are recovered per-image below.
    return orig, idx, stride   # idx & stride placeholders — resolved per-image


def run_counting_merged(model, img_dir, lbl_dir, conf):
    """
    Counting eval that merges tile detections back to per-original-image counts.

    For MCD (ground) images (stems ending in _G): single image, evaluated directly.
    For UAV tiles (stems matching *_t????_U): grouped by original stem, detections
        mapped to full-image coords, cross-tile NMS applied, boxes summed.

    GT counts for UAV originals are reconstructed by summing tile-label box counts
    (each box belongs to exactly one tile because MIN_VIS_FRAC=0.30 — a box may
    appear in multiple tiles when it straddles the overlap zone, but it is only
    assigned to the tile where it passes the visibility threshold first).
    Actually, due to 30% overlap a tassel can appear in up to 2 tiles.
    So gt_count per original = count distinct boxes across all tiles, which
    equals the raw count from the original annotation (that's the ground truth).
    We therefore load GT from the *original* UAV XML annotations directly.
    """
    import re

    # ── Ground (MCD) images: direct eval ────────────────────────────────────
    mcd_gt, mcd_pred, mcd_stems = [], [], []
    for img_p in sorted(set(f for p in IMG_PATS
                            for f in glob.glob(os.path.join(img_dir, p)))):
        stem = Path(img_p).stem
        if not stem.endswith('_G'):
            continue
        lbl_p = os.path.join(lbl_dir, stem + '.txt')
        if not os.path.exists(lbl_p): continue
        with open(lbl_p) as f:
            gt_cnt = len([l for l in f if l.strip()])
        result = model.predict(img_p, conf=conf,
                               max_det=300, device=DEVICE, verbose=False)[0]
        mcd_gt.append(gt_cnt)
        mcd_pred.append(len(result.boxes))
        mcd_stems.append(stem)

    # ── UAV tiles: group → full-image merge ─────────────────────────────────
    # Collect all tile files grouped by original image stem
    tile_pat = re.compile(r'^(.+)_t(\d{4})_U$')
    tile_groups = collections.defaultdict(list)   # orig_stem → [(tile_path, tile_idx)]
    for img_p in sorted(set(f for p in IMG_PATS
                            for f in glob.glob(os.path.join(img_dir, p)))):
        stem = Path(img_p).stem
        m = tile_pat.match(stem)
        if not m: continue
        orig  = m.group(1)
        t_idx = int(m.group(2))
        tile_groups[orig].append((img_p, t_idx))

    # Also need original GT box count per UAV image.
    # This is the number of REAL annotations in the original .mat / .xml file,
    # NOT the sum of tile labels (which double-count boxes in overlap zones).
    # We stored it implicitly: the tile label files together represent all GT
    # boxes with MIN_VIS_FRAC filtering, so some boxes may be in >1 tile.
    # The cleanest source of truth is the original XML annotation files.
    uav_gt_from_xml = {}
    if UAV_DET_LBL_TEST:
        for xml_p in glob.glob(os.path.join(UAV_DET_LBL_TEST, '*.xml')):
            orig_stem = Path(xml_p).stem
            boxes = load_voc_xml(xml_p, 1, 1)   # W/H=1 → we only need count
            uav_gt_from_xml[orig_stem] = len(boxes)

    stride = int(TILE_SIZE * (1 - TILE_OVERLAP))
    uav_gt, uav_pred, uav_stems = [], [], []

    for orig_stem, tile_list in sorted(tile_groups.items()):
        if orig_stem not in uav_gt_from_xml:
            continue
        gt_cnt = uav_gt_from_xml[orig_stem]

        # Build lookup: tile_idx → (tile_image_path, label_dir)
        idx_to_path = {t: p for p, t in tile_list}

        tile_pred_total = 0
        for t_idx, tile_path in sorted(idx_to_path.items()):
            # ── Load exact offset written at tiling time ──────────────────
            # Labels are mirrored into DATASET_DIR/labels/test/ during Cell 6.
            lbl_dir_test = os.path.join(DATASET_DIR, 'labels', 'test')
            offset_path  = os.path.join(lbl_dir_test,
                                        Path(tile_path).stem + '.offset')
            if not os.path.exists(offset_path):
                # Fallback: offset file missing (e.g. dataset rebuilt without
                # re-running Cell 5).  Skip this tile with a warning.
                print(f'  [WARN] missing offset for {Path(tile_path).stem} — re-run Cell 5')
                continue
            with open(offset_path) as f:
                x0, y0 = map(int, f.read().split())

            result = model.predict(tile_path, conf=conf,
                                   max_det=300, device=DEVICE, verbose=False)[0]
            if len(result.boxes) == 0:
                continue

            bxy = result.boxes.xyxy.cpu().numpy()   # (N,4) tile-pixel coords

            # Ownership filter: keep only detections whose centre lies in the
            # non-overlapping owned strip [0, stride) × [0, stride) of this
            # tile.  Each tassel's centre falls in exactly one tile's owned
            # strip, so no cross-tile NMS is needed.
            cx = (bxy[:,0] + bxy[:,2]) / 2
            cy = (bxy[:,1] + bxy[:,3]) / 2
            owned = (cx < stride) & (cy < stride)
            tile_pred_total += int(owned.sum())

        uav_gt.append(gt_cnt)
        uav_pred.append(tile_pred_total)
        uav_stems.append(orig_stem + '_U_merged')

    # ── Combine both domains ─────────────────────────────────────────────────
    gt_l    = mcd_gt    + uav_gt
    pred_l  = mcd_pred  + uav_pred
    stems   = mcd_stems + uav_stems
    return gt_l, pred_l, stems, mcd_gt, mcd_pred, uav_gt, uav_pred


# ── mAP (tile-level, valid for box metrics) ──────────────────────────────────
print('Running detection eval (mAP on tiles)…')
vr=model_eval.val(data=YAML_PATH,split='test',conf=EVAL_CONF, max_det=300,
                  device=DEVICE,verbose=False,plots=True,name='eval_test')
map50=float(vr.box.map50); map5095=float(vr.box.map)
prec=float(vr.box.mp); rec=float(vr.box.mr)
f1=2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
print(f'  mAP@0.5      : {map50:.4f}')
print(f'  mAP@0.5:0.95 : {map5095:.4f}')
print(f'  Precision    : {prec:.4f}')
print(f'  Recall       : {rec:.4f}')
print(f'  F1           : {f1:.4f}')

# ── Counting eval (per-original-image, correct unit) ─────────────────────────
print('\nRunning counting eval (per-original-image, tiles merged)…')
gt_l,pred_l,stems,mcd_gt,mcd_pred,uav_gt,uav_pred = run_counting_merged(
    model_eval,
    os.path.join(DATASET_DIR,'images','test'),
    os.path.join(DATASET_DIR,'labels','test'),
    EVAL_CONF)

if not gt_l:
    print('❌ No test images matched. Check DATASET_DIR and test labels.')
else:
    mae,rmse,r2,gt_arr,pred_arr=counting_metrics(gt_l,pred_l)
    print(f'  Original images evaluated: {len(gt_l)} '
          f'(MCD:{len(mcd_gt)}  UAV:{len(uav_gt)})')
    print(f'  MAE    : {mae:.3f}  (target <2.0)  {"✅" if mae<2.0 else "❌"}')
    print(f'  RMSE   : {rmse:.3f}')
    print(f'  R²     : {r2:.4f}  (target >0.97)  {"✅" if r2>0.97 else "❌"}')

    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5),facecolor='#0d1117')
    lim=max(gt_arr.max(),pred_arr.max())*1.1+1
    ax1.set_facecolor('#161b22')
    ax1.plot([0,lim],[0,lim],'--',color='#555',lw=1.5)
    # colour points by domain
    mcd_n = len(mcd_gt)
    ax1.scatter(gt_arr[:mcd_n],  pred_arr[:mcd_n],  c='#00FF41', alpha=0.75,
                edgecolors='#0d1117', s=55, label='Ground (MCD)')
    ax1.scatter(gt_arr[mcd_n:],  pred_arr[mcd_n:],  c='#00BFFF', alpha=0.75,
                edgecolors='#0d1117', s=55, label='UAV (merged)')
    if len(gt_arr)>2:
        z=np.polyfit(gt_arr,pred_arr,1); xs=np.linspace(0,lim,100)
        ax1.plot(xs,np.poly1d(z)(xs),'#FF6B35',lw=2)
    ax1.legend(facecolor='#1e2530', labelcolor='white', fontsize=9)
    ax1.set_title(f'Test  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}',
                  color='white',fontsize=11,fontfamily='monospace')
    ax1.set_xlabel('GT Count',color='#e6edf3'); ax1.set_ylabel('Pred Count',color='#e6edf3')
    ax1.tick_params(colors='#888')
    for sp in ax1.spines.values(): sp.set_color('#30363d')

    diff=pred_arr-gt_arr
    ax2.set_facecolor('#161b22')
    ax2.hist(diff,bins=20,color='#00FF41',edgecolor='#0d1117',alpha=0.85)
    ax2.axvline(0,color='#FF6B35',lw=2,linestyle='--')
    ax2.set_title(f'Error Distribution  mean={diff.mean():.2f}  std={diff.std():.2f}',
                  color='white',fontsize=11,fontfamily='monospace')
    ax2.set_xlabel('Error (pred−gt)',color='#e6edf3'); ax2.set_ylabel('Images',color='#e6edf3')
    ax2.tick_params(colors='#888')
    for sp in ax2.spines.values(): sp.set_color('#30363d')

    fig.suptitle('YOLO26 — Maize Tassel Evaluation (per-original-image)',
                 color='white',fontsize=14,fontfamily='monospace')
    plt.tight_layout()
    plt.savefig('evaluation.png',dpi=130,bbox_inches='tight',facecolor=fig.get_facecolor())
    plt.show()

    with open('per_image_counts.csv','w',newline='') as f:
        w=csv.writer(f); w.writerow(['stem','gt','pred','error','abs_error','domain'])
        for stem,gt_v,pred_v in zip(stems,gt_l,pred_l):
            dom = 'Ground' if stem.endswith('_G') else 'UAV'
            w.writerow([stem,gt_v,pred_v,pred_v-gt_v,abs(pred_v-gt_v),dom])

    if mcd_gt:
        g_mae,g_rmse,g_r2,_,_=counting_metrics(mcd_gt,mcd_pred)
        print(f'  Ground domain ({len(mcd_gt)} imgs): MAE={g_mae:.3f}  RMSE={g_rmse:.3f}  R²={g_r2:.4f}')
    if uav_gt:
        u_mae,u_rmse,u_r2,_,_=counting_metrics(uav_gt,uav_pred)
        print(f'  UAV domain   ({len(uav_gt)} imgs): MAE={u_mae:.3f}  RMSE={u_rmse:.3f}  R²={u_r2:.4f}')

    print('\n  Searching optimal conf threshold (0.20–0.60)…')
    best_conf_thr, best_mae_thr = EVAL_CONF, mae
    for thr in [round(x*0.05,2) for x in range(4,13)]:
        gt_t,pred_t,_,_,_,_,_=run_counting_merged(model_eval,
            os.path.join(DATASET_DIR,'images','test'),
            os.path.join(DATASET_DIR,'labels','test'), thr)
        if not gt_t: continue
        m,_,_,_,_=counting_metrics(gt_t,pred_t)
        print(f'    conf={thr:.2f} → MAE={m:.3f}')
        if m < best_mae_thr: best_mae_thr=m; best_conf_thr=thr
    print(f'  ✅ Best conf threshold: {best_conf_thr:.2f}  →  MAE={best_mae_thr:.3f}')
    print(f'     (update EVAL_CONF_THR in Cell 2 config if different from {EVAL_CONF_THR})')

    print(f'\n══ FINAL RESULTS ════════════════════════════════════════')
    print(f'   mAP@0.5 : {map50:.4f}  |  F1 : {f1:.4f}')
    print(f'   MAE     : {mae:.3f}   |  RMSE : {rmse:.3f}  |  R² : {r2:.4f}')
    print(f'   Best conf threshold found: {best_conf_thr:.2f}  (MAE={best_mae_thr:.3f})')
    print(f'═════════════════════════════════════════════════════════')


---
## 🔍 Cell 12 — Inference

UAV images in `test/` are stored as 640×640 tiles.  For visualisation and
counting this cell groups them back to their original image by:
  1. Running per-tile inference.
  2. Mapping each box back to full-image pixel coordinates using the tile's
     exact (x0, y0) offset loaded from the companion `.offset` file written
     at tiling time — no grid-shape estimation needed.
  3. Applying cross-tile NMS on the full-image coordinate space to remove
     duplicate detections in the 30% overlap zones.
  4. Displaying the merged result and reporting one count per original image.

MCD ground images are single-tile; they go through a direct predict path.


In [ ]:
import numpy as np, glob, os, csv, collections, re
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path
from ultralytics import YOLO

PREDICT_SOURCE = os.path.join(DATASET_DIR, 'images', 'test')
PRED_CONF      = 0.25
# max_det=300 matches YOLO26's one-to-one head training cap (default).
# Dense UAV tiles can have 50-150 tassels/640px tile — 300 is sufficient at 30% row spacing.
# If counts are unexpectedly low on very dense fields, uncomment the line below
# and retrain with end2end=False (one-to-many head + NMS) for higher max_det support:
# PRED_CONF_FALLBACK: model.predict(..., max_det=500, end2end=False, iou=NMS_IOU)
MAX_SHOW       = 6           # number of original images to visualise

model_pred = YOLO(BEST_WEIGHTS)
os.makedirs('predictions', exist_ok=True)

# ── Collect all test images and group UAV tiles by original stem ──────────
tile_pat  = re.compile(r'^(.+)_t(\d{4})_U$')
tile_groups   = collections.defaultdict(list)   # orig_stem → [(img_path, tile_idx)]
ground_images = []                               # (img_path, stem) for MCD

for img_p in sorted(set(f for p in IMG_PATS
                        for f in glob.glob(os.path.join(PREDICT_SOURCE, p)))):
    stem = Path(img_p).stem
    m = tile_pat.match(stem)
    if m:
        tile_groups[m.group(1)].append((img_p, int(m.group(2))))
    else:
        ground_images.append((img_p, stem))

print(f'Ground images : {len(ground_images)}')
print(f'UAV originals : {len(tile_groups)} (from {sum(len(v) for v in tile_groups.values())} tiles)')
print(f'Running inference…')

stride      = int(TILE_SIZE * (1 - TILE_OVERLAP))
all_results = []   # list of dict(stem, count, domain)
vis_shown   = 0

# ── MCD ground images ─────────────────────────────────────────────────────
for img_p, stem in ground_images:
    result = model_pred.predict(img_p, conf=PRED_CONF,
                                max_det=300, device=DEVICE, verbose=False)[0]
    boxes = result.boxes.xyxy.cpu().numpy() if len(result.boxes) else np.zeros((0,4))
    confs = result.boxes.conf.cpu().numpy() if len(result.boxes) else np.array([])
    count = len(boxes)
    all_results.append({'stem':stem,'count':count,'domain':'Ground'})
    print(f'  [Ground] {stem[:50]:<50} → {count} tassels')

    if vis_shown < MAX_SHOW:
        try:
            img = np.array(Image.open(img_p).convert('RGB'))
        except Exception as e:
            print(f'  [WARN] {stem}: {e}'); continue
        H,W = img.shape[:2]
        fig,ax = plt.subplots(figsize=(14,7),facecolor='#0d1117')
        ax.imshow(img); ax.axis('off')
        for box,cf in zip(boxes,confs):
            x1,y1,x2,y2=box
            col=(float(max(0,1-cf)), float(min(1,cf*1.5)), 0.1)
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.8,
                         edgecolor=col,facecolor=col,alpha=0.09))
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.8,
                         edgecolor=col,facecolor='none',alpha=0.9))
        ax.text(22,max(58,int(H*0.05)),f'  [Ground]  Tassels: {count}  ',
                fontsize=15,color='white',fontweight='bold',fontfamily='monospace',
                bbox=dict(facecolor='#161b22',edgecolor='#00FF41',
                          boxstyle='round,pad=0.5',linewidth=2))
        ax.set_title(f'{stem}  |  {W}×{H} px',color='white',fontsize=11,
                     fontfamily='monospace',backgroundcolor='#161b22',pad=6)
        fig.savefig(f'predictions/{stem}_pred.jpg',dpi=110,
                    bbox_inches='tight',facecolor=fig.get_facecolor())
        plt.show(); plt.close()
        vis_shown += 1

# ── UAV tiled images: merge tiles → full-image detections ────────────────
for orig_stem, tile_list in sorted(tile_groups.items()):
    full_boxes  = []   # (x1,y1,x2,y2) in full-image coords
    full_confs  = []
    # offset_map: tile_path → (x0, y0) loaded from .offset file
    offset_map  = {}
    lbl_dir_test = os.path.join(DATASET_DIR, 'labels', 'test')
    for tile_path, t_idx in tile_list:
        offset_path = os.path.join(lbl_dir_test, Path(tile_path).stem + '.offset')
        if not os.path.exists(offset_path):
            print(f'  [WARN] missing offset for {Path(tile_path).stem} — re-run Cell 5')
            continue
        with open(offset_path) as _f:
            _x0, _y0 = map(int, _f.read().split())
        offset_map[tile_path] = (_x0, _y0)

    for tile_path, t_idx in tile_list:
        if tile_path not in offset_map:
            continue   # offset missing — already warned above
        x0, y0 = offset_map[tile_path]

        result = model_pred.predict(tile_path, conf=PRED_CONF,
                                    max_det=300, device=DEVICE, verbose=False)[0]
        if len(result.boxes) == 0:
            continue
        bxy   = result.boxes.xyxy.cpu().numpy()
        bconf = result.boxes.conf.cpu().numpy()

        # Map tile-pixel coords → full-image coords using exact stored offset
        bxy_full = bxy.copy()
        bxy_full[:, 0] += x0;  bxy_full[:, 2] += x0
        bxy_full[:, 1] += y0;  bxy_full[:, 3] += y0

        # Ownership filter: keep only detections whose centre lies in the
        # non-overlapping owned strip [0, stride) × [0, stride) of this tile.
        # Each tassel's centre falls in exactly one tile's owned strip, so no
        # cross-tile NMS is needed.
        cx = (bxy[:,0]+bxy[:,2])/2
        cy = (bxy[:,1]+bxy[:,3])/2
        owned = (cx < stride) & (cy < stride)
        full_boxes.append(bxy_full[owned])
        full_confs.append(bconf[owned])

    if full_boxes:
        all_bxy   = np.concatenate(full_boxes, axis=0)
        all_bconf = np.concatenate(full_confs, axis=0)
        # Final cross-tile IoU-NMS pass in full-image space.
        # Note: YOLO26's internal one-to-one head eliminates intra-tile duplicates;
        # this NMS only handles the inter-tile overlap zone where the same tassel
        # may appear in two adjacent tiles' owned strips after offset-mapping.
        keep = _nms_boxes(all_bxy, all_bconf, NMS_IOU)
        final_boxes = all_bxy[keep]
        final_confs = all_bconf[keep]
    else:
        final_boxes = np.zeros((0,4))
        final_confs = np.array([])

    count = len(final_boxes)
    all_results.append({'stem': orig_stem+'_U_merged', 'count': count, 'domain': 'UAV'})
    print(f'  [UAV]    {orig_stem[:50]:<50} → {count} tassels'
          f'  ({len(tile_list)} tiles merged)')

    if vis_shown < MAX_SHOW:
        # Stitch tiles into a canvas for visualisation using exact offsets
        canvas_W = max(x0 + TILE_SIZE for x0, _ in offset_map.values()) if offset_map else TILE_SIZE
        canvas_H = max(y0 + TILE_SIZE for _, y0 in offset_map.values()) if offset_map else TILE_SIZE
        canvas   = np.zeros((canvas_H, canvas_W, 3), dtype=np.uint8)
        for tile_path, t_idx in tile_list:
            if tile_path not in offset_map:
                continue
            x0c, y0c = offset_map[tile_path]
            try:
                t_arr = np.array(Image.open(tile_path).convert('RGB'))
            except Exception:
                continue
            th = min(TILE_SIZE, canvas_H - y0c)
            tw = min(TILE_SIZE, canvas_W - x0c)
            canvas[y0c:y0c+th, x0c:x0c+tw] = t_arr[:th, :tw]

        H, W = canvas.shape[:2]
        fig, ax = plt.subplots(figsize=(16, 8), facecolor='#0d1117')
        ax.imshow(canvas); ax.axis('off')
        for box, cf in zip(final_boxes, final_confs):
            x1,y1,x2,y2 = box
            col_box = (float(max(0,1-cf)), float(min(1,cf*1.5)), 0.1)
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,
                         edgecolor=col_box,facecolor=col_box,alpha=0.09))
            ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,lw=1.5,
                         edgecolor=col_box,facecolor='none',alpha=0.9))
        ax.text(22, max(58, int(H*0.05)),
                f'  [UAV merged]  Tassels: {count}  ({len(tile_list)} tiles)  ',
                fontsize=14,color='white',fontweight='bold',fontfamily='monospace',
                bbox=dict(facecolor='#161b22',edgecolor='#FF6B35',
                          boxstyle='round,pad=0.5',linewidth=2))
        ax.set_title(f'{orig_stem}  |  approx {W}×{H} px (stitched)',
                     color='white',fontsize=11,fontfamily='monospace',
                     backgroundcolor='#161b22',pad=6)
        fig.savefig(f'predictions/{orig_stem}_merged_pred.jpg',dpi=100,
                    bbox_inches='tight',facecolor=fig.get_facecolor())
        plt.show(); plt.close()
        vis_shown += 1

# ── Summary ───────────────────────────────────────────────────────────────
counts = [r['count'] for r in all_results]
if counts:
    print(f'\n── Summary ── {len(counts)} original images  |  {sum(counts)} tassels total')
    print(f'   Mean/img:{np.mean(counts):.1f}  Min:{min(counts)}  Max:{max(counts)}')

with open('predictions/counts.csv','w',newline='') as f:
    w = csv.writer(f); w.writerow(['image','count','domain'])
    for r in all_results:
        w.writerow([r['stem'], r['count'], r['domain']])
print('✅ predictions/counts.csv saved')


---
## 💾 Cell 13 — Save Everything to Google Drive


In [ ]:
import shutil, os, zipfile
from google.colab import files

items = [
    (BEST_WEIGHTS,                              'best_weights.pt'),
    (f'{WORK_DIR}/runs/detect/{S3_NAME}/weights/last.pt', 'last_weights_s3.pt'),
    (f'{WORK_DIR}/runs/detect/{S3_NAME}/results.png',     'stage3_curves.png'),
    (f'{WORK_DIR}/runs/detect/{S2_NAME}/results.png',     'stage2_curves.png'),
    (f'{WORK_DIR}/runs/detect/{S1_NAME}/results.png',     'stage1_curves.png'),
    ('evaluation.png',                          'evaluation.png'),
    ('per_image_counts.csv',                    'per_image_counts.csv'),
    ('predictions/counts.csv',                  'inference_counts.csv'),
    (YAML_PATH,                                 'dataset.yaml'),
]
print(f'Saving to: {DRIVE_RESULTS}')
saved=0
for src, dst_name in items:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(DRIVE_RESULTS, dst_name))
        print(f'  ✅ {dst_name}')
        saved+=1
    else:
        print(f'  ⚠️  not found: {src}')

print('\nArchiving runs/detect to Drive…')
with zipfile.ZipFile(os.path.join(DRIVE_RESULTS,'runs_detect.zip'),'w',zipfile.ZIP_DEFLATED) as zf:
    runs_root=os.path.join(WORK_DIR,'runs','detect')
    if os.path.isdir(runs_root):
        for root,dirs,fnames in os.walk(runs_root):
            for fname in fnames:
                full=os.path.join(root,fname)
                zf.write(full,os.path.relpath(full,WORK_DIR))
print('  ✅ runs_detect.zip archived')
print(f'\n✅ {saved}/{len(items)} items saved to Drive')
write_status(S3_NAME,'SAVED','All outputs saved to Drive')

print('\nDownloading best.pt to your PC…')
try:
    files.download(BEST_WEIGHTS)
except Exception as e:
    print(f'  ⚠️  Download failed (tab in background?): {e}')
    print(f'  Weights safe at Drive: {DRIVE_RESULTS}/best_weights.pt')


In [ ]:
import shutil
import os

run_dir = "/content/maize_work/runs/detect/maize_yolo26_s3_final"
drive_results = "/content/drive/MyDrive/maize_yolo26_results_confirm_final"

for fname in ["last.pt", "best.pt"]:
    src = os.path.join(run_dir, "weights", fname)
    if os.path.exists(src):
        shutil.copy2(
            src,
            os.path.join(drive_results, f"maize_yolo26_s3_final_{fname}")
        )

        if fname == "last.pt":
            shutil.copy2(src, os.path.join(drive_results, "LATEST_last.pt"))

        if fname == "best.pt":
            shutil.copy2(src, os.path.join(drive_results, "LATEST_best.pt"))

print("✅ Manual backup completed")

✅ Manual backup completed


## Download best.pt for Backend

The trained model is saved to your Google Drive:
**maize_yolo26_results/best.pt**

### To use it locally:
1. Download best.pt from your Google Drive
2. Place it in: 
3. Restart the Flask backend - real detection will be enabled automatically
4. No code changes needed - inference.py loads it on startup
